In [1]:
import vrep 
import sys
import time 
import numpy as np
from tank import *

In [29]:
vrep.simxFinish(-1) # closes all opened connections, in case any prevoius wasnt finished
clientID=vrep.simxStart('127.0.0.1',19999,True,True,5000,5) # start a connection

if clientID!=-1:
    print ("Connected to remote API server")
else:
    print("Not connected to remote API server")
    sys.exit("Could not connect")

#create instance of Tank
tank=Tank(clientID)

# get handle to proximity sensor
err_code,ps_handle = vrep.simxGetObjectHandle(clientID,"Proximity_sensor", vrep.simx_opmode_blocking)

import skfuzzy as fuzz
from skfuzzy import control as ctrl

def create_fuzzy_controller(mode='lagodny'):
    distance = ctrl.Antecedent(np.arange(0, 10.01, 0.01), 'distance')
    speed = ctrl.Consequent(np.arange(0, 30.01, 0.01), 'speed')
    speed0 = 0
    
    if mode == 'lagodny':
        speed0 = 30
        distance['blisko'] = fuzz.trimf(distance.universe, [0, 0, 0.7])
        distance['srednio'] = fuzz.trimf(distance.universe, [0.5, 3, 6])
        distance['daleko'] = fuzz.trimf(distance.universe, [4, 8, 10])

        speed['wolno'] = fuzz.trimf(speed.universe, [0, 0, 1])
        speed['srednio'] = fuzz.trimf(speed.universe, [0.8, 10, 26])
        speed['szybko'] = fuzz.trimf(speed.universe, [26, 28, 30])
    
    elif mode == 'agresywny':
        speed0 = 60
        distance['blisko'] = fuzz.trimf(distance.universe, [0, 0.2, 0.6])
        distance['srednio'] = fuzz.trimf(distance.universe, [0.5, 2, 3])
        distance['daleko'] = fuzz.trimf(distance.universe, [2.5, 3, 10])

        speed['wolno'] = fuzz.trimf(speed.universe, [0, 0, 6])
        speed['srednio'] = fuzz.trimf(speed.universe, [5, 25, 26])
        speed['szybko'] = fuzz.trimf(speed.universe, [22, 50, 60])

    rule1 = ctrl.Rule(distance['blisko'], speed['wolno'])
    rule2 = ctrl.Rule(distance['srednio'], speed['srednio'])
    rule3 = ctrl.Rule(distance['daleko'], speed['szybko'])

    system = ctrl.ControlSystem([rule1, rule2, rule3])
    sim = ctrl.ControlSystemSimulation(system)

    return sim, speed0
_, _, _, _, _ = vrep.simxReadProximitySensor(clientID, ps_handle, vrep.simx_opmode_streaming)

sim, start_speed = create_fuzzy_controller('lagodny')
print(f"Symulacja na prędkości początkowej: {start_speed}")

t = time.time()
counter = 0
dist_stop = 0.5

tank.go()
tank.leftvelocity = start_speed
tank.rightvelocity = start_speed
tank.setVelocity()
print(f"Check velocity: {tank.leftvelocity}")
while (time.time() - t) < 10:
    err_code, detectionState, detectedPoint, detectedObjectHandle, detectedSurfaceNormalVector = vrep.simxReadProximitySensor(clientID, ps_handle, vrep.simx_opmode_buffer)
    
    if err_code == vrep.simx_return_ok and detectionState:
        distance = np.linalg.norm(detectedPoint)
    else:
        distance = 10
    
    sim.input['distance'] = distance
    # print(f"Distance: {distance}")
    try:
        sim.compute()
        speed = sim.output['speed']
    except Exception as e:
        print(e)
        speed = 0
        
    tank.go()
    tank.leftvelocity = speed
    tank.rightvelocity = speed
    tank.setVelocity()
    
    if counter % 1 == 0:
        print(f"Distance: {distance:.2f}, Speed (fuzzy): {speed:.2f}, Left velocity: {tank.leftvelocity:.2f}, Right velocity: {tank.rightvelocity:.2f}")

    counter += 1
    time.sleep(0.1)

vrep.simxStopSimulation(clientID, vrep.simx_opmode_oneshot) # stop the simulation in vrep

Connected to remote API server
Symulacja na prędkości początkowej: 30
Check velocity: 30
'speed'
Distance: 10.00, Speed (fuzzy): 0.00, Left velocity: 0.00, Right velocity: 0.00
Distance: 7.61, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Distance: 7.33, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Distance: 7.33, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Distance: 7.39, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Distance: 7.48, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Distance: 7.45, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Distance: 7.43, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Distance: 7.16, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Distance: 6.97, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Distance: 6.77, Speed (fuzzy): 28.00, Left velocity: 28.00, Right velocity: 28.00
Dis

1